# Feshchenko/Friedrichs angle verification — probe geometry
Verifies the analytic angle table for x2 independent, corr(x1,x3)=rho:
the nonzero Friedrichs angles form a 4-cycle {1}-{3}-{12}-{23}-{1} with every edge = rho;
all other pairs are 0 (nested, independence, or conditional independence).

**Scope of the verification.** Over all seven nonempty subsets (the full set {123} is nested
with everything and contributes only an eigenvalue 1), Delta = I - C has exact spectrum
{1-2rho, 1, 1, 1, 1, 1, 1+2rho}. Under this convention **A2 (positive definiteness) holds
iff rho < 1/2** — this notebook does NOT claim A2 for all rho < 1. The runs at rho = 0.6
and rho = 0.9 are **expected A2 failures**: they demonstrate that the Feshchenko sufficient
condition is conservative in the Gaussian regime, where identifiability itself provably holds
up to rho = 1 (closed form F(rho), hermite_verification artifact).

Estimator: fully out-of-sample split-half canonical correlation with per-estimate standard errors
(degree-2 features for the zero checks; heavy-tail SE inflation at higher degrees is expected and documented).
Convention status: **CONFIRMED** against Idrissi et al. (arXiv 2310.06567, Definition 3): unit diagonal,
-c(L2(sigma_A), L2(sigma_B)) off-diagonal, indexed by the FULL power set (including the empty set, which is
nested in everything and adds one unit eigenvalue). Spectrum over all 2^3 = 8 subsets:
{1-2rho, 1, 1, 1, 1, 1, 1, 1+2rho}. The rho >= 1/2 A2 failure stands under their exact definition.

Outputs to `MyDrive/KDD_Interactions/results/feshchenko_check/`.


In [ ]:
# Cell 1 — Mount Drive and set up output folder
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/KDD_Interactions'
OUT = os.path.join(BASE, 'results', 'feshchenko_check')
os.makedirs(OUT, exist_ok=True)
print('output folder:', OUT)


In [ ]:
# Cell 2 — OOS Friedrichs-angle estimator with standard errors
import numpy as np, json, csv, time, hashlib
from itertools import product

def herm_norm_feats(x, deg):
    H = [np.ones_like(x), x]
    for n in range(1, deg):
        H.append((x * H[n] - np.sqrt(n) * H[n - 1]) / np.sqrt(n + 1))
    return np.column_stack(H)

def subspace_feats(sample, A, deg, rows=slice(None)):
    n = len(next(iter(sample.values()))[rows])
    Hs = {j: herm_norm_feats(sample[j][rows], deg) for j in A}
    cols = []
    for d in product(range(deg + 1), repeat=len(A)):
        if sum(d) == 0: continue
        col = np.ones(n)
        for j, dj in zip(A, d):
            col = col * Hs[j][:, dj]
        cols.append(col)
    return np.column_stack(cols)

def friedrichs_oos_se(sample, A, B, deg=2, int_deg=None, eig_tol=2e-2):
    """Split-half out-of-sample Friedrichs angle estimate with standard error.
    Residualization betas, whitening, and canonical directions are all fit on
    the train half; the correlation is evaluated on the test half."""
    if int_deg is None: int_deg = 2 * deg
    C = tuple(sorted(set(A) & set(B)))
    n = len(next(iter(sample.values())))
    half = n // 2
    parts = {}
    for name, sl in [("tr", slice(0, half)), ("te", slice(half, n))]:
        U = subspace_feats(sample, A, deg, sl)
        V = subspace_feats(sample, B, deg, sl)
        m = U.shape[0]
        W = np.column_stack([np.ones(m)] + ([subspace_feats(sample, C, int_deg, sl)] if C else []))
        parts[name] = (U, V, W)
    Utr, Vtr, Wtr = parts["tr"]; Ute, Vte, Wte = parts["te"]
    bU, *_ = np.linalg.lstsq(Wtr, Utr, rcond=None)
    bV, *_ = np.linalg.lstsq(Wtr, Vtr, rcond=None)
    Utr, Vtr = Utr - Wtr @ bU, Vtr - Wtr @ bV
    Ute, Vte = Ute - Wte @ bU, Vte - Wte @ bV
    def dirs(M):
        G = M.T @ M / M.shape[0]
        w, Q = np.linalg.eigh(G)
        keep = w > eig_tol * w.max()
        return Q[:, keep] / np.sqrt(w[keep])
    Tu, Tv = dirs(Utr), dirs(Vtr)
    Cx = Tu.T @ (Utr.T @ Vtr / Utr.shape[0]) @ Tv
    uu, s, vv = np.linalg.svd(Cx)
    a, b = Tu @ uu[:, 0], Tv @ vv[0, :]
    ut, vt = Ute @ a, Vte @ b
    den = np.sqrt(np.mean(ut ** 2) * np.mean(vt ** 2))
    c = float(np.mean(ut * vt) / den)
    se = float(np.std(ut * vt) / np.sqrt(len(ut)) / den)
    return c, se

CODE_SHA = hashlib.sha256(b"".join(f.__code__.co_code for f in
    [herm_norm_feats, subspace_feats, friedrichs_oos_se])).hexdigest()
print("code sha256:", CODE_SHA)


In [ ]:
# Cell 3 — Angle table, Delta spectrum, checks; results to Drive
N = 1_500_000
RHOS = [0.3, 0.6, 0.9]
SUBS = [(), (1,), (2,), (3,), (1, 2), (1, 3), (2, 3), (1, 2, 3)]  # FULL power set per Idrissi et al. Def. 3;
# the empty set and {123} are nested with every other subset (angle 0) and each adds one unit eigenvalue
RHO_EDGES = [frozenset([(1,), (3,)]), frozenset([(1,), (2, 3)]),
             frozenset([(3,), (1, 2)]), frozenset([(1, 2), (2, 3)])]

rng = np.random.default_rng(11)
rows, checks = [], []
t0 = time.time()
for rho in RHOS:
    x1, x2, z = rng.standard_normal((3, N))
    x3 = rho * x1 + np.sqrt(1 - rho ** 2) * z
    S = {1: x1, 2: x2, 3: x3}
    vals = {}
    for i in range(len(SUBS)):
        for j in range(i + 1, len(SUBS)):
            A, B = SUBS[i], SUBS[j]
            if set(A) <= set(B) or set(B) <= set(A):
                continue                                # nested: angle 0 by definition
            key = frozenset([A, B])
            expected = rho if key in RHO_EDGES else 0.0
            c, se = friedrichs_oos_se(S, A, B, deg=2)
            vals[key] = c
            rows.append({"experiment": "feshchenko_check", "rho": rho,
                         "A": str(A), "B": str(B), "estimate": c, "se": se,
                         "expected": expected})
    D = np.eye(len(SUBS))
    for i in range(len(SUBS)):
        for j in range(i + 1, len(SUBS)):
            A, B = SUBS[i], SUBS[j]
            key = frozenset([A, B])
            D[i, j] = D[j, i] = -vals.get(key, 0.0)
    ev = np.linalg.eigvalsh(D)
    a2_pd = bool(ev.min() > 1e-8)
    a2_expected = bool(rho < 0.5)
    rows.append({"experiment": "feshchenko_check", "rho": rho, "A": "spectrum", "B": "",
                 "estimate": float(ev.min()), "se": float(ev.max()),
                 "expected": 1 - 2 * rho})
    rows.append({"experiment": "feshchenko_check", "rho": rho, "A": "a2_status", "B": "",
                 "estimate": float(a2_pd), "se": 0.0, "expected": float(a2_expected)})
    status = "A2 holds (PD)" if a2_pd else "A2 FAILS (expected for rho >= 1/2)"
    print(f"rho={rho}: eigenvalues {np.round(ev, 4)}  predicted extremes {1-2*rho:+.4f}/{1+2*rho:.4f}"
          f"  -> {status}  ({time.time()-t0:5.1f}s)", flush=True)

edge_ok = all(abs(r["estimate"] - r["expected"]) < 6 * max(r["se"], 1e-4)
              for r in rows if r["A"] != "spectrum" and r["expected"] > 0)
zero_ok = all(abs(r["estimate"]) < max(4 * r["se"], 0.01)
              for r in rows if r["A"] != "spectrum" and r["expected"] == 0)
spec_ok = all(abs(r["estimate"] - r["expected"]) < 0.03
              for r in rows if r["A"] == "spectrum")
a2_ok = all(r["estimate"] == r["expected"] for r in rows if r["A"] == "a2_status")
checks = [("four rho-edges detected at expected values", edge_ok),
          ("all zero angles consistent with 0 (deg-2 features)", zero_ok),
          ("lambda_min(Delta) matches 1-2rho within 0.03", spec_ok),
          ("A2 PD status matches rho<1/2 prediction (0.3: PD; 0.6, 0.9: expected failures)", a2_ok)]

import csv, json, os
with open(os.path.join(OUT, "results.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
with open(os.path.join(OUT, "metadata.json"), "w") as f:
    json.dump({"experiment": "feshchenko_check", "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "N": N, "rhos": RHOS, "code_sha256": CODE_SHA, "numpy": np.__version__,
               "analytic_spectrum": "{1-2rho, 1, 1, 1, 1, 1, 1, 1+2rho} over the full power set (8 subsets); PD iff rho < 1/2",
               "a2_interpretation": "rho=0.6 and rho=0.9 are expected A2 failures demonstrating conservatism of the sufficient condition",
               "convention_status": "CONFIRMED: matches Idrissi et al. arXiv 2310.06567 Definition 3 (maximal coalitional precision matrix) verbatim"},
              f, indent=2)
story = []
for name, ok in checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    story.append(line); print(line)
with open(os.path.join(OUT, "check.txt"), "w") as f:
    f.write("\n".join(story) + "\n")
print("\nwrote results.csv, metadata.json, check.txt")
